# Experiment 6 — Part 3: Sequence-to-Sequence Learning
### CS3807 Deep Learning Laboratory — Encoder–Decoder LSTM on a synthetic sequence-reversal task
**Additional Exercise #7:** output sequence of a *different length* from the input (reverse, then drop the last element: 5 -> 4).

`Input Sequence -> Encoder LSTM -> Context State (h, c) -> Decoder LSTM -> Output Sequence`
Training uses teacher forcing; all reported metrics come from greedy autoregressive inference.

## 1. Imports, configuration, seeds

In [1]:
import os, json, time, random, warnings
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ---------------- configuration ----------------
N_SAMPLES   = 8000        # synthetic sequences
MIN_TOKEN, MAX_TOKEN = 1, 20   # integer token range
IN_LEN      = 5           # input sequence length
OUT_LEN_A   = 5           # task A: pure reversal
OUT_LEN_B   = 4           # task B (Ex. #7): reversal with the last element removed
START_TOKEN = 0           # index 0 is reserved as the decoder start token
VOCAB       = MAX_TOKEN + 1    # tokens 0..20
EMB_DIM     = 32
HIDDEN      = 64
LR          = 1e-3
BATCH_SIZE  = 64
MAX_EPOCHS  = 30
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15

OUT = "experiment6_part3_outputs"
DIRS = {k: os.path.join(OUT, k) for k in ["plots", "tables", "model_results"]}
for d in DIRS.values(): os.makedirs(d, exist_ok=True)

plt.rcParams.update({
    "figure.facecolor":"white","axes.facecolor":"white","savefig.facecolor":"white",
    "font.size":12,"axes.titlesize":15,"axes.labelsize":13,
    "xtick.labelsize":11,"ytick.labelsize":11,"legend.fontsize":11,
    "axes.grid":True,"grid.alpha":0.3,"axes.spines.top":False,"axes.spines.right":False})

GENERATED_PNGS = []
def save_fig(fig, name):
    p = os.path.join(DIRS["plots"], name)
    fig.tight_layout(); fig.savefig(p, dpi=300, bbox_inches="tight"); plt.close(fig)
    GENERATED_PNGS.append(p); print("saved:", p); return p

print("TensorFlow", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs available:", len(gpus), gpus if gpus else "(running on CPU)")

TensorFlow 2.20.0
GPUs available: 2 [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


## 2. Synthetic dataset generation (reversal, and reversal-minus-last)

In [2]:
def make_dataset(n, in_len, out_len, seed=SEED):
    # X: random integer sequences (tokens MIN_TOKEN..MAX_TOKEN), unique so no duplicate leaks across splits.
    # Y: reverse(X) truncated to out_len  ->  out_len == in_len gives pure reversal (Task A)
    #                                         out_len == in_len-1 drops the final reversed element (Task B)
    rng = np.random.RandomState(seed)
    seen, rows = set(), []
    while len(rows) < n:
        s = tuple(rng.randint(MIN_TOKEN, MAX_TOKEN + 1, size=in_len))
        if s in seen: continue
        seen.add(s); rows.append(s)
    X = np.array(rows, dtype="int32")
    Y = X[:, ::-1][:, :out_len].copy().astype("int32")
    return X, Y

def split_data(X, Y):
    n = len(X); idx = np.random.RandomState(SEED).permutation(n)
    a, b = int(TRAIN_FRAC * n), int((TRAIN_FRAC + VAL_FRAC) * n)
    tr, va, te = idx[:a], idx[a:b], idx[b:]
    assert not (set(tr) & set(va)) and not (set(tr) & set(te)) and not (set(va) & set(te))
    return (X[tr], Y[tr]), (X[va], Y[va]), (X[te], Y[te])

def decoder_io(Y):
    # teacher forcing: decoder input = [START, y1, ..., y_{T-1}], decoder target = [y1, ..., y_T]
    dec_in = np.concatenate([np.full((len(Y), 1), START_TOKEN, dtype="int32"), Y[:, :-1]], axis=1)
    return dec_in, Y

def describe(tag, X, Y, tr, va, te):
    print("\n--- " + tag + " ---")
    print("Token range (values)   :", MIN_TOKEN, "to", MAX_TOKEN, "| vocabulary size (incl. START=0):", VOCAB)
    print("Input sequence length  :", X.shape[1], "| output sequence length:", Y.shape[1])
    print("Total samples          :", len(X), "(all input sequences unique)")
    print("Train X/Y shapes       :", tr[0].shape, tr[1].shape)
    print("Validation X/Y shapes  :", va[0].shape, va[1].shape)
    print("Test X/Y shapes        :", te[0].shape, te[1].shape)
    print("Example  input         :", X[0].tolist())
    print("Example  target        :", Y[0].tolist())

XA, YA = make_dataset(N_SAMPLES, IN_LEN, OUT_LEN_A, seed=SEED)
trA, vaA, teA = split_data(XA, YA)
describe("TASK A: reversal (5 -> 5)", XA, YA, trA, vaA, teA)

XB, YB = make_dataset(N_SAMPLES, IN_LEN, OUT_LEN_B, seed=SEED + 1)
trB, vaB, teB = split_data(XB, YB)
describe("TASK B (Ex. #7): reversal with last element removed (5 -> 4)", XB, YB, trB, vaB, teB)
print("\nTask B transformation: target = reverse(input)[:4]  e.g. " +
      str(XB[0].tolist()) + " -> " + str(YB[0].tolist()))


--- TASK A: reversal (5 -> 5) ---
Token range (values)   : 1 to 20 | vocabulary size (incl. START=0): 21
Input sequence length  : 5 | output sequence length: 5
Total samples          : 8000 (all input sequences unique)
Train X/Y shapes       : (5600, 5) (5600, 5)
Validation X/Y shapes  : (1200, 5) (1200, 5)
Test X/Y shapes        : (1200, 5) (1200, 5)
Example  input         : [7, 20, 15, 11, 8]
Example  target        : [8, 11, 15, 20, 7]

--- TASK B (Ex. #7): reversal with last element removed (5 -> 4) ---
Token range (values)   : 1 to 20 | vocabulary size (incl. START=0): 21
Input sequence length  : 5 | output sequence length: 4
Total samples          : 8000 (all input sequences unique)
Train X/Y shapes       : (5600, 5) (5600, 4)
Validation X/Y shapes  : (1200, 5) (1200, 4)
Test X/Y shapes        : (1200, 5) (1200, 4)
Example  input         : [5, 1, 18, 17, 20]
Example  target        : [20, 17, 18, 1]

Task B transformation: target = reverse(input)[:4]  e.g. [5, 1, 18, 17, 20] -> [2

## 3. Encoder–decoder LSTM (training model + greedy inference models)

In [3]:
def build_seq2seq(name):
    # ---- encoder ----
    enc_in = layers.Input(shape=(None,), dtype="int32", name="encoder_input")
    enc_emb = layers.Embedding(VOCAB, EMB_DIM, name="encoder_embedding")(enc_in)
    _, h, c = layers.LSTM(HIDDEN, return_state=True, name="encoder_lstm")(enc_emb)
    states = [h, c]                       # context state
    # ---- decoder ----
    dec_in = layers.Input(shape=(None,), dtype="int32", name="decoder_input")
    dec_emb_layer = layers.Embedding(VOCAB, EMB_DIM, name="decoder_embedding")
    dec_lstm = layers.LSTM(HIDDEN, return_sequences=True, return_state=True, name="decoder_lstm")
    dec_dense = layers.Dense(VOCAB, activation="softmax", name="decoder_output")
    dec_out, _, _ = dec_lstm(dec_emb_layer(dec_in), initial_state=states)
    model = keras.Model([enc_in, dec_in], dec_dense(dec_out), name=name)
    model.compile(optimizer=keras.optimizers.Adam(LR),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    # ---- inference graphs (share the trained weights) ----
    enc_model = keras.Model(enc_in, states, name=name + "_encoder")
    sh, sc = layers.Input(shape=(HIDDEN,)), layers.Input(shape=(HIDDEN,))
    d_out, d_h, d_c = dec_lstm(dec_emb_layer(dec_in), initial_state=[sh, sc])
    dec_model = keras.Model([dec_in, sh, sc], [dec_dense(d_out), d_h, d_c], name=name + "_decoder")
    return model, enc_model, dec_model

def greedy_decode(enc_model, dec_model, X, out_len):
    # autoregressive inference: no teacher forcing, the model consumes its own previous prediction
    h, c = enc_model.predict(X, batch_size=256, verbose=0)
    tok = np.full((len(X), 1), START_TOKEN, dtype="int32")
    preds = np.zeros((len(X), out_len), dtype="int32")
    for t in range(out_len):
        probs, h, c = dec_model.predict([tok, h, c], batch_size=256, verbose=0)
        tok = probs[:, -1, :].argmax(-1).astype("int32").reshape(-1, 1)
        preds[:, t] = tok[:, 0]
    return preds

def token_and_sequence_accuracy(pred, true):
    token_acc = float((pred == true).mean())
    seq_acc = float(np.all(pred == true, axis=1).mean())   # a sequence counts only if EVERY token matches
    return token_acc, seq_acc

## 4. Train both models

In [4]:
def run_task(tag, key, tr, va, te, out_len):
    tf.keras.utils.set_random_seed(SEED)
    model, enc_model, dec_model = build_seq2seq(key)
    with open(os.path.join(DIRS["model_results"], key + "_model_summary.txt"), "w") as f:
        model.summary(print_fn=lambda s: f.write(s + "\n"))
    model.summary()
    params = int(sum(np.prod(w.shape) for w in model.trainable_weights))

    dtr_in, dtr_out = decoder_io(tr[1]); dva_in, dva_out = decoder_io(va[1])
    t0 = time.time()
    hist = model.fit([tr[0], dtr_in], dtr_out,
                     validation_data=([va[0], dva_in], dva_out),
                     epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, verbose=2)
    train_time = time.time() - t0
    h = pd.DataFrame(hist.history); h.insert(0, "epoch", np.arange(1, len(h) + 1))
    h.to_csv(os.path.join(DIRS["model_results"], key + "_training_history.csv"), index=False)

    pred = greedy_decode(enc_model, dec_model, te[0], out_len)
    tok_acc, seq_acc = token_and_sequence_accuracy(pred, te[1])
    res = {"Task": tag, "Input Length": tr[0].shape[1], "Output Length": out_len,
           "Token Accuracy (%)": round(tok_acc * 100, 2), "Sequence Accuracy (%)": round(seq_acc * 100, 2),
           "Final Training Loss": round(float(h["loss"].iloc[-1]), 6),
           "Final Validation Loss": round(float(h["val_loss"].iloc[-1]), 6),
           "Best Validation Loss": round(float(h["val_loss"].min()), 6),
           "Parameters": params, "Epochs": len(h), "Training Time (s)": round(train_time, 2)}
    print("\n===== " + tag + " =====")
    for k, v in res.items(): print("  " + k + ": " + str(v))
    return model, h, pred, res

modelA, histA, predA, resA = run_task("Task A: reversal (5 -> 5)", "seq2seq", trA, vaA, teA, OUT_LEN_A)

I0000 00:00:1789916710.377511      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789916710.380891      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "seq2seq"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_input       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, None, 32)  │        672 │ encoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, None, 32)  │        672 │ decoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 64),      │     24,832 │ encoder_embeddin… │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │     24,832 │ decoder_embeddin… │
│                     │ 64), (None, 64),  │            │ encoder_lstm[0][… │
│                     │ (None, 64)]       │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_output      │ (None, None, 21)  │      1,365 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 52,373 (204.58 KB)

 Trainable params: 52,373 (204.58 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
88/88 - 7s - 75ms/step - accuracy: 0.1624 - loss: 2.8113 - val_accuracy: 0.2413 - val_loss: 2.3622
Epoch 2/30
88/88 - 1s - 9ms/step - accuracy: 0.3231 - loss: 2.0245 - val_accuracy: 0.4432 - val_loss: 1.6303
Epoch 3/30
88/88 - 1s - 9ms/step - accuracy: 0.6216 - loss: 1.2324 - val_accuracy: 0.7750 - val_loss: 0.8805
Epoch 4/30
88/88 - 1s - 9ms/step - accuracy: 0.8595 - loss: 0.6694 - val_accuracy: 0.9138 - val_loss: 0.5055
Epoch 5/30
88/88 - 1s - 9ms/step - accuracy: 0.9431 - loss: 0.4041 - val_accuracy: 0.9558 - val_loss: 0.3306
Epoch 6/30
88/88 - 1s - 9ms/step - accuracy: 0.9690 - loss: 0.2744 - val_accuracy: 0.9742 - val_loss: 0.2362
Epoch 7/30
88/88 - 1s - 9ms/step - accuracy: 0.9823 - loss: 0.1986 - val_accuracy: 0.9828 - val_loss: 0.1786
Epoch 8/30
88/88 - 1s - 9ms/step - accuracy: 0.9889 - loss: 0.1529 - val_accuracy: 0.9877 - val_loss: 0.1407
Epoch 9/30
88/88 - 1s - 9ms/step - accuracy: 0.9931 - loss: 0.1187 - val_accuracy: 0.9922 - val_loss: 0.1113
Epoch 10/30
88/88 

In [5]:
modelB, histB, predB, resB = run_task("Task B (Ex. #7): different length (5 -> 4)",
                                      "different_length", trB, vaB, teB, OUT_LEN_B)

Model: "different_length"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_input       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, None, 32)  │        672 │ encoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, None, 32)  │        672 │ decoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 64),      │     24,832 │ encoder_embeddin… │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │     24,832 │ decoder_embeddin… │
│                     │ 64), (None, 64),  │            │ encoder_lstm[0][… │
│                     │ (None, 64)]       │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_output      │ (None, None, 21)  │      1,365 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 52,373 (204.58 KB)

 Trainable params: 52,373 (204.58 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
88/88 - 3s - 35ms/step - accuracy: 0.1849 - loss: 2.8149 - val_accuracy: 0.2608 - val_loss: 2.3443
Epoch 2/30
88/88 - 1s - 8ms/step - accuracy: 0.3305 - loss: 2.0070 - val_accuracy: 0.4346 - val_loss: 1.6451
Epoch 3/30
88/88 - 1s - 8ms/step - accuracy: 0.6252 - loss: 1.2281 - val_accuracy: 0.8058 - val_loss: 0.8659
Epoch 4/30
88/88 - 1s - 8ms/step - accuracy: 0.8901 - loss: 0.6251 - val_accuracy: 0.9415 - val_loss: 0.4567
Epoch 5/30
88/88 - 1s - 8ms/step - accuracy: 0.9646 - loss: 0.3487 - val_accuracy: 0.9750 - val_loss: 0.2795
Epoch 6/30
88/88 - 1s - 8ms/step - accuracy: 0.9844 - loss: 0.2225 - val_accuracy: 0.9848 - val_loss: 0.2001
Epoch 7/30
88/88 - 1s - 8ms/step - accuracy: 0.9919 - loss: 0.1537 - val_accuracy: 0.9921 - val_loss: 0.1422
Epoch 8/30
88/88 - 1s - 9ms/step - accuracy: 0.9961 - loss: 0.1122 - val_accuracy: 0.9946 - val_loss: 0.1057
Epoch 9/30
88/88 - 1s - 8ms/step - accuracy: 0.9981 - loss: 0.0860 - val_accuracy: 0.9960 - val_loss: 0.0843
Epoch 10/30
88/88 

## 5. Training plots

In [6]:
def plot_history(h, key, title_prefix, fn_loss, fn_acc):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(h["epoch"], h["loss"], "o-", color="#1f77b4", lw=2, ms=3.5, label="Training loss")
    ax.plot(h["epoch"], h["val_loss"], "s--", color="#d62728", lw=2, ms=3.5, label="Validation loss")
    ax.set_xlabel("Epoch", fontweight="bold"); ax.set_ylabel("Sparse Categorical Crossentropy", fontweight="bold")
    ax.set_title(title_prefix + ": Training vs Validation Loss", fontweight="bold"); ax.legend()
    save_fig(fig, fn_loss)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(h["epoch"], h["accuracy"] * 100, "o-", color="#2a9d8f", lw=2, ms=3.5, label="Training token accuracy")
    ax.plot(h["epoch"], h["val_accuracy"] * 100, "s--", color="#e76f51", lw=2, ms=3.5, label="Validation token accuracy")
    ax.set_xlabel("Epoch", fontweight="bold"); ax.set_ylabel("Token Accuracy (%)", fontweight="bold")
    ax.set_title(title_prefix + ": Training vs Validation Token Accuracy (teacher forcing)", fontweight="bold")
    ax.legend()
    save_fig(fig, fn_acc)

plot_history(histA, "seq2seq", "Seq2Seq Reversal (5 -> 5)",
             "01_seq2seq_training_loss.png", "02_seq2seq_training_accuracy.png")
plot_history(histB, "different_length", "Different-Length Seq2Seq (5 -> 4)",
             "03_different_length_training_loss.png", "04_different_length_training_accuracy.png")

saved: experiment6_part3_outputs/plots/01_seq2seq_training_loss.png
saved: experiment6_part3_outputs/plots/02_seq2seq_training_accuracy.png
saved: experiment6_part3_outputs/plots/03_different_length_training_loss.png
saved: experiment6_part3_outputs/plots/04_different_length_training_accuracy.png


## 6. Actual test predictions (at least five per task)

In [7]:
def examples_table(te, pred, csv_name, n=8):
    rows = [{"Sample": i + 1,
             "Input Sequence": str(te[0][i].tolist()),
             "Expected Output": str(te[1][i].tolist()),
             "Predicted Output": str(pred[i].tolist()),
             "Correct": bool(np.array_equal(pred[i], te[1][i]))} for i in range(n)]
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(DIRS["tables"], csv_name), index=False)
    print(df.to_string(index=False))
    return df

print("TASK A - reversal (held-out test set, greedy model predictions)")
exA = examples_table(teA, predA, "01_seq2seq_test_examples.csv")
print("\nTASK B - different length 5 -> 4 (held-out test set, greedy model predictions)")
exB = examples_table(teB, predB, "02_different_length_test_examples.csv")

pd.DataFrame([resA]).to_csv(os.path.join(DIRS["tables"], "03_seq2seq_metrics.csv"), index=False)
pd.DataFrame([resB]).to_csv(os.path.join(DIRS["tables"], "04_different_length_metrics.csv"), index=False)
pd.DataFrame([resA, resB]).to_csv(os.path.join(DIRS["tables"], "03_different_length_metrics.csv"), index=False)

TASK A - reversal (held-out test set, greedy model predictions)
 Sample      Input Sequence     Expected Output    Predicted Output  Correct
      1 [18, 12, 19, 13, 7] [7, 13, 19, 12, 18] [7, 13, 19, 12, 18]     True
      2 [12, 16, 19, 8, 17] [17, 8, 19, 16, 12] [17, 8, 19, 16, 12]     True
      3 [17, 14, 16, 14, 6] [6, 14, 16, 14, 17] [6, 14, 16, 14, 17]     True
      4    [9, 11, 4, 2, 6]    [6, 2, 4, 11, 9]    [6, 2, 4, 11, 9]     True
      5  [19, 6, 4, 11, 10]  [10, 11, 4, 6, 19]  [10, 11, 4, 6, 19]     True
      6  [1, 19, 19, 4, 19]  [19, 4, 19, 19, 1]  [19, 4, 19, 19, 1]     True
      7  [6, 2, 16, 11, 12]  [12, 11, 16, 2, 6]  [12, 11, 16, 2, 6]     True
      8  [17, 3, 14, 3, 13]  [13, 3, 14, 3, 17]  [13, 3, 14, 3, 17]     True

TASK B - different length 5 -> 4 (held-out test set, greedy model predictions)
 Sample      Input Sequence Expected Output Predicted Output  Correct
      1  [10, 8, 14, 16, 3]  [3, 16, 14, 8]   [3, 16, 14, 8]     True
      2  [2, 14, 18, 19

## 7. Auto-generated factual report text

In [8]:
def wrong_token_stats(pred, true):
    wrong_seq = ~np.all(pred == true, axis=1)
    n_wrong = int(wrong_seq.sum())
    if n_wrong == 0: return n_wrong, 0.0, np.zeros(true.shape[1])
    mean_err = float((pred[wrong_seq] != true[wrong_seq]).sum(1).mean())
    per_pos = (pred != true).mean(0)
    return n_wrong, mean_err, per_pos

L = []
L.append("EXPERIMENT 6 - PART 3: SEQUENCE-TO-SEQUENCE LEARNING (auto-generated from this run)")
L.append("TensorFlow " + tf.__version__ + " | GPUs available: " + str(len(gpus)) + " | seed " + str(SEED))
L.append("")
L.append("DATASET (synthetic, no external data)")
L.append("Token range: integers " + str(MIN_TOKEN) + "-" + str(MAX_TOKEN) +
         "; index 0 is reserved as the decoder START token, so vocabulary size = " + str(VOCAB) + ".")
L.append("Samples per task: " + str(N_SAMPLES) + ", all input sequences unique; split 70/15/15 by index permutation " +
         "(train " + str(len(trA[0])) + ", validation " + str(len(vaA[0])) + ", test " + str(len(teA[0])) +
         "), disjoint indices verified, so no test leakage.")
L.append("Task A: input length " + str(IN_LEN) + " -> output length " + str(OUT_LEN_A) +
         "; target = reverse(input). Example " + str(XA[0].tolist()) + " -> " + str(YA[0].tolist()) + ".")
L.append("Task B (Additional Exercise #7): input length " + str(IN_LEN) + " -> output length " + str(OUT_LEN_B) +
         "; target = reverse(input) with the final element of the reversed sequence removed. Example " +
         str(XB[0].tolist()) + " -> " + str(YB[0].tolist()) + ".")
L.append("")
L.append("MODEL ARCHITECTURE (identical for both tasks)")
L.append("Encoder: Embedding(" + str(VOCAB) + ", " + str(EMB_DIM) + ") -> LSTM(" + str(HIDDEN) +
         ", return_state=True); the final (h, c) pair is the context state.")
L.append("Decoder: Embedding(" + str(VOCAB) + ", " + str(EMB_DIM) + ") -> LSTM(" + str(HIDDEN) +
         ", initial_state = context) -> Dense(" + str(VOCAB) + ", softmax), one token per time step.")
L.append("Training uses teacher forcing (decoder input = START + shifted target); evaluation uses greedy " +
         "autoregressive decoding where the decoder consumes its own previous prediction.")
L.append("Optimizer Adam, learning rate " + str(LR) + ", sparse categorical crossentropy, batch size " +
         str(BATCH_SIZE) + ", " + str(MAX_EPOCHS) + " epochs.")
L.append("The decoder is run for exactly the required number of output steps, so output length matches the target " +
         "length (" + str(OUT_LEN_A) + " for Task A, " + str(OUT_LEN_B) + " for Task B).")
L.append("")
for r, pred, te, ex in [(resA, predA, teA, exA), (resB, predB, teB, exB)]:
    L.append("RESULTS - " + r["Task"])
    L.append("Trainable parameters: " + str(r["Parameters"]) + "; epochs " + str(r["Epochs"]) +
             "; training time " + str(r["Training Time (s)"]) + " s.")
    L.append("Final training loss " + str(r["Final Training Loss"]) + "; final validation loss " +
             str(r["Final Validation Loss"]) + "; best validation loss " + str(r["Best Validation Loss"]) + ".")
    L.append("Test token accuracy " + str(r["Token Accuracy (%)"]) + "%; test sequence accuracy " +
             str(r["Sequence Accuracy (%)"]) + "% (computed from greedy predictions on " +
             str(len(te[0])) + " held-out sequences).")
    nw, me, pp = wrong_token_stats(pred, te[1])
    L.append("Sequences with at least one wrong token: " + str(nw) + "/" + str(len(te[0])) +
             "; average wrong tokens within those sequences: " + str(round(me, 3)) +
             "; per-position token error rate: " + str([round(float(v), 4) for v in pp]) + ".")
    L.append("Five actual test examples (input -> expected | predicted):")
    for _, row in ex.head(5).iterrows():
        L.append("  " + row["Input Sequence"] + " -> " + row["Expected Output"] +
                 " | " + row["Predicted Output"] + " | correct=" + str(row["Correct"]))
    L.append("")
L.append("TOKEN ACCURACY VS SEQUENCE ACCURACY")
L.append("Token accuracy = correctly predicted tokens / total tokens; sequence accuracy = fully correct sequences " +
         "/ total sequences. A sequence is counted only when every one of its tokens matches.")
L.append("Task A: token accuracy " + str(resA["Token Accuracy (%)"]) + "% vs sequence accuracy " +
         str(resA["Sequence Accuracy (%)"]) + "% (difference " +
         str(round(resA["Token Accuracy (%)"] - resA["Sequence Accuracy (%)"], 2)) + " points).")
L.append("Task B: token accuracy " + str(resB["Token Accuracy (%)"]) + "% vs sequence accuracy " +
         str(resB["Sequence Accuracy (%)"]) + "% (difference " +
         str(round(resB["Token Accuracy (%)"] - resB["Sequence Accuracy (%)"], 2)) + " points).")
L.append("Sequence accuracy is the stricter measure because errors are not averaged: a single wrong token out of " +
         str(OUT_LEN_A) + " invalidates the whole sequence, and errors that are spread across different sequences " +
         "damage sequence accuracy far more than token accuracy.")
L.append("")
L.append("DIFFERENT-LENGTH OBSERVATIONS (Additional Exercise #7)")
L.append("Changing the output length only changes the number of decoder time steps; the encoder-decoder graph is " +
         "unchanged, which shows that the context state, not the input length, determines what the decoder generates.")
L.append("Task B produces " + str(OUT_LEN_B) + " tokens from " + str(IN_LEN) + " inputs, so the model must also " +
         "learn to ignore the first input element (which never appears in the target).")
L.append("Parameter counts: Task A " + str(resA["Parameters"]) + ", Task B " + str(resB["Parameters"]) +
         " (difference " + str(resA["Parameters"] - resB["Parameters"]) + ").")

txt = "\n".join(L)
with open(os.path.join(DIRS["model_results"], "part3_auto_generated_report_text.txt"), "w") as f:
    f.write(txt)
print(txt)

EXPERIMENT 6 - PART 3: SEQUENCE-TO-SEQUENCE LEARNING (auto-generated from this run)
TensorFlow 2.20.0 | GPUs available: 2 | seed 42

DATASET (synthetic, no external data)
Token range: integers 1-20; index 0 is reserved as the decoder START token, so vocabulary size = 21.
Samples per task: 8000, all input sequences unique; split 70/15/15 by index permutation (train 5600, validation 1200, test 1200), disjoint indices verified, so no test leakage.
Task A: input length 5 -> output length 5; target = reverse(input). Example [7, 20, 15, 11, 8] -> [8, 11, 15, 20, 7].
Task B (Additional Exercise #7): input length 5 -> output length 4; target = reverse(input) with the final element of the reversed sequence removed. Example [5, 1, 18, 17, 20] -> [20, 17, 18, 1].

MODEL ARCHITECTURE (identical for both tasks)
Encoder: Embedding(21, 32) -> LSTM(64, return_state=True); the final (h, c) pair is the context state.
Decoder: Embedding(21, 32) -> LSTM(64, initial_state = context) -> Dense(21, softmax), 

## 8. Final summary

In [9]:
print("=" * 78)
print("EXPERIMENT 6 - PART 3 + ADDITIONAL EXERCISE #7 : FINAL SUMMARY")
print("=" * 78)
summary = pd.DataFrame([resA, resB])
print(summary.to_string(index=False))
print("-" * 78)
print("Samples per task        :", N_SAMPLES, "| train/val/test:", len(trA[0]), "/", len(vaA[0]), "/", len(teA[0]))
print("Token range             :", MIN_TOKEN, "-", MAX_TOKEN, "| vocabulary size:", VOCAB)
print("Sequence lengths        : Task A", IN_LEN, "->", OUT_LEN_A, "| Task B", IN_LEN, "->", OUT_LEN_B)
print("Trainable parameters    : Task A", resA["Parameters"], "| Task B", resB["Parameters"])
print("Token accuracy (%)      : Task A", resA["Token Accuracy (%)"], "| Task B", resB["Token Accuracy (%)"])
print("Sequence accuracy (%)   : Task A", resA["Sequence Accuracy (%)"], "| Task B", resB["Sequence Accuracy (%)"])
print("Final train/val loss    : Task A", resA["Final Training Loss"], "/", resA["Final Validation Loss"],
      "| Task B", resB["Final Training Loss"], "/", resB["Final Validation Loss"])
print("Output folder           :", os.path.abspath(OUT))
print("Generated PNG files:")
for p in GENERATED_PNGS: print("   ", p)
print("Other output files:")
for root, _, files in os.walk(OUT):
    for fn in sorted(files):
        if not fn.endswith(".png"): print("   ", os.path.join(root, fn))
print("=" * 78)

EXPERIMENT 6 - PART 3 + ADDITIONAL EXERCISE #7 : FINAL SUMMARY
                                      Task  Input Length  Output Length  Token Accuracy (%)  Sequence Accuracy (%)  Final Training Loss  Final Validation Loss  Best Validation Loss  Parameters  Epochs  Training Time (s)
                 Task A: reversal (5 -> 5)             5              5               99.82                  99.42             0.008303               0.013630              0.013630       52373      30              28.67
Task B (Ex. #7): different length (5 -> 4)             5              4               99.94                  99.75             0.005644               0.009735              0.009735       52373      30              24.81
------------------------------------------------------------------------------
Samples per task        : 8000 | train/val/test: 5600 / 1200 / 1200
Token range             : 1 - 20 | vocabulary size: 21
Sequence lengths        : Task A 5 -> 5 | Task B 5 -> 4
Trainable parameter

In [10]:
import shutil

shutil.make_archive(
    "/kaggle/working/experiment6_part3_outputs",
    "zip",
    "/kaggle/working/experiment6_part3_outputs"
)

print("Created: /kaggle/working/experiment6_part3_outputs.zip")

Created: /kaggle/working/experiment6_part3_outputs.zip
